# Gaussian Processes

_Classical ML_

**A distribution over functions: predictions come with honest error bars.**

Most models give you one prediction. A Gaussian Process (GP) gives a prediction and its uncertainty.

     Instead of fitting one curve, a GP keeps a whole probability distribution over all curves that could fit.

---

This notebook builds a Gaussian Process up one step at a time: first fit one on tiny clean data and probe its uncertainty, then fit it on real diabetes measurements and watch the error bars widen where data is sparse. Run each cell, read the note above it, then experiment. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

A Gaussian Process fits a whole distribution over functions, so every prediction comes with an honest uncertainty. We build it in three steps: (1) make a small noisy dataset, (2) fit the GP, (3) probe its uncertainty near and far from the data.

### Step 1 — Make a small noisy dataset

We sample 12 input points in `[-3, 3]` and set their targets to `sin(x)` plus a little Gaussian noise. Keeping it tiny makes the GP's behavior easy to read: it will be confident near these points and unsure far from them.

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

rng = np.random.RandomState(0)

X = np.sort(rng.uniform(-3, 3, 12)).reshape(-1, 1)   # 12 input points
y = np.sin(X).ravel() + 0.1 * rng.randn(12)          # noisy sine targets

print("X shape:", X.shape, " y shape:", y.shape)

### Step 2 — Choose a kernel and fit the GP

The **kernel** encodes our assumptions about the function. Here it is an `RBF` (smooth, nearby points are correlated) plus a `WhiteKernel` (independent observation noise). Fitting tunes the kernel's hyperparameters by maximizing the log marginal likelihood — a single number measuring how well the GP explains the data.

In [ ]:
kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.01)

gp = GaussianProcessRegressor(kernel=kernel, random_state=0,
                              normalize_y=True).fit(X, y)

print("learned kernel:", gp.kernel_)
print("log marginal likelihood:", round(gp.log_marginal_likelihood_value_, 3))

### Step 3 — Probe uncertainty near vs far from the data

We ask the GP to predict at two points: `x=0` (surrounded by training data) and `x=10` (far outside it). Each prediction returns a mean and a standard deviation. The key GP property: the std is small where data is dense and grows large where there is none.

In [ ]:
Xtest = np.array([[0.0], [10.0]])   # one point near data, one far away

mean, std = gp.predict(Xtest, return_std=True)

print("at x=0  : mean=%.3f  std=%.3f" % (mean[0], std[0]))
print("at x=10 : mean=%.3f  std=%.3f" % (mean[1], std[1]))
print("-> std is far larger where there is no data")

## Visualize the data & results

_Fitting diabetes progression from BMI: how confident is the model where data is sparse?_

### Step 1 — Take a real feature and subsample it

We switch to the diabetes dataset and use the single **BMI** feature against disease progression. We draw a random subset of 40 points so some regions of the BMI axis are denser than others — that uneven coverage is exactly what makes the GP's varying confidence visible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, RBF, WhiteKernel)

dia = load_diabetes()
x = dia.data[:, 2]            # BMI feature (mean-centered, scaled)
y = dia.target.astype(float)

rng = np.random.RandomState(0)
idx = np.sort(rng.choice(len(x), 40, replace=False))   # 40 random points

X = x[idx].reshape(-1, 1)
yo = y[idx]
print("training points:", X.shape[0])

### Step 2 — Fit the GP on the real data

We use the same kind of kernel — a constant-scaled `RBF` for smoothness plus a `WhiteKernel` for noise. Real measurements are noisier than the toy sine, so the white-noise term matters more here. `normalize_y=True` centers the targets so the GP's zero-mean prior is sensible.

In [ ]:
kernel = (ConstantKernel(1.0) * RBF(length_scale=0.1)
          + WhiteKernel(noise_level=1.0))

gp = GaussianProcessRegressor(kernel=kernel, random_state=0,
                              normalize_y=True).fit(X, yo)

print("learned kernel:", gp.kernel_)

### Step 3 — Plot the mean curve and its confidence band

We predict across a dense grid of BMI values and draw the mean prediction plus a ±2-std band (roughly a 95% confidence region). Where training points cluster, the band is tight; where they thin out, the band flares — the GP is telling you exactly where to distrust it.

In [ ]:
xs = np.linspace(x.min() - 0.02, x.max() + 0.02, 50).reshape(-1, 1)
mean, std = gp.predict(xs, return_std=True)

plt.scatter(X.ravel(), yo, c="#ffb454", s=30, zorder=3)
plt.plot(xs.ravel(), mean, c="#4ea1ff")
plt.fill_between(xs.ravel(), mean - 2 * std, mean + 2 * std,
                 color="#7ee787", alpha=0.3)
plt.title("GP regression on Diabetes (BMI vs progression)")
plt.xlabel("BMI (mean-centered, scaled)")
plt.ylabel("disease progression")
plt.show()